In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from lifelines.utils import concordance_index

emb = pd.read_csv("patient_embeddings_with_labels.csv")  # cruk_id + embedding columns (+ maybe shorter_dfs_balanced)
tx = pd.read_csv("data/tracerX.csv")
tx.columns = tx.columns.str.strip()

# Keep essential survival columns
tx_keep = tx[['cruk_id', 'dfs_time', 'cens_dfs']].copy()

# Merge embeddings with survival
df = emb.merge(tx_keep, on='cruk_id', how='inner')

# Basic cleaning
df = df.dropna(subset=['dfs_time', 'cens_dfs'])
df = df[df['dfs_time'] > 0]  # Cox requires positive durations

# Features (all non-ID/non-target columns that came from embeddings)
feature_cols = [c for c in df.columns if c not in ['cruk_id', 'dfs_time', 'cens_dfs', 'shorter_dfs_balanced']]
X = df[feature_cols].values.astype(np.float32)
T = df['dfs_time'].values.astype(np.float32)
E = df['cens_dfs'].values.astype(np.float32)  # you confirmed 1 = event, 0 = censored

# Train/val/test split (stratify on event status)
X_train, X_tmp, T_train, T_tmp, E_train, E_tmp = train_test_split(
    X, T, E, test_size=0.3, random_state=42, stratify=E
)
X_val, X_test, T_val, T_test, E_val, E_test = train_test_split(
    X_tmp, T_tmp, E_tmp, test_size=0.5, random_state=42, stratify=E_tmp
)

# Standardize features (fit on train only)
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype(np.float32)
X_val   = scaler.transform(X_val).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)

# Torch tensors
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def to_tensor(arr, dtype=torch.float32):
    return torch.tensor(arr, dtype=dtype)

Xt = to_tensor(X_train).to(device)
Xv = to_tensor(X_val).to(device)
Xte = to_tensor(X_test).to(device)
Tt = to_tensor(T_train).to(device)
Tv = to_tensor(T_val).to(device)
Tte = to_tensor(T_test).to(device)
Et = to_tensor(E_train).to(device)
Ev = to_tensor(E_val).to(device)
Ete = to_tensor(E_test).to(device)

train_loader = DataLoader(TensorDataset(Xt, Tt, Et), batch_size=32, shuffle=True)
val_loader   = DataLoader(TensorDataset(Xv, Tv, Ev), batch_size=256, shuffle=False)

# DeepSurv model (MLP → risk score)
class DeepSurvMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)  # risk score (no activation)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)  # shape: (batch,)

model = DeepSurvMLP(input_dim=X_train.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Cox partial log-likelihood loss
# (assumes E=1 event, E=0 censored)
def cox_ph_loss(risk, time, event):
    # Sort by descending time so at-risk set is cumulative
    order = torch.argsort(time, descending=True)
    r = risk[order]
    e = event[order]

    # log cumulative sum exp(risk)
    log_cumsum_exp = torch.logcumsumexp(r, dim=0)
    # Contributions only from events
    losses = (r - log_cumsum_exp) * e
    # Negative partial log-likelihood (to minimize)
    return -losses.sum() / torch.clamp(e.sum(), min=1.0)

# Training loop with early stopping
best_val_c = -np.inf
patience = 10
pat = 0
epochs = 100

for epoch in range(1, epochs+1):
    model.train()
    train_loss = 0.0
    for xb, tb, eb in train_loader:
        optimizer.zero_grad()
        risk = model(xb)
        loss = cox_ph_loss(risk, tb, eb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation: compute risk on full val set, c-index
    model.eval()
    with torch.no_grad():
        val_risk = model(Xv)
        # C-index expects higher risk => shorter survival; we use raw risk
        c_index_val = concordance_index(T_val, (-val_risk).cpu().numpy(), Ev.cpu().numpy())  # negate for shorter time = higher risk
        val_loss = cox_ph_loss(val_risk, Tv, Ev).item()

    print(f"Epoch {epoch:03d} | TrainLoss {train_loss/len(train_loader):.4f} | "
          f"ValLoss {val_loss:.4f} | Val C-index {c_index_val:.4f}")

    # Early stopping on C-index
    if c_index_val > best_val_c + 1e-4:
        best_val_c = c_index_val
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        pat = 0
    else:
        pat += 1
        if pat >= patience:
            print("Early stopping.")
            break

# Load best
model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

# Final evaluation on test set
model.eval()
with torch.no_grad():
    test_risk = model(Xte)
    c_index_test = concordance_index(T_test, (-test_risk).cpu().numpy(), Ete.cpu().numpy())
print(f"Test C-index: {c_index_test:.4f}")

# Save model and scaler
torch.save(model.state_dict(), "deepsurv_model.pt")
import joblib; joblib.dump(scaler, "embedding_scaler.joblib")

Epoch 001 | TrainLoss 3.0038 | ValLoss 3.5515 | Val C-index 0.4541
Epoch 002 | TrainLoss 2.9903 | ValLoss 3.5598 | Val C-index 0.4462
Epoch 003 | TrainLoss 2.9860 | ValLoss 3.5742 | Val C-index 0.4492
Epoch 004 | TrainLoss 2.9657 | ValLoss 3.5849 | Val C-index 0.4648
Epoch 005 | TrainLoss 2.9648 | ValLoss 3.5975 | Val C-index 0.4902
Epoch 006 | TrainLoss 2.9641 | ValLoss 3.6077 | Val C-index 0.4951
Epoch 007 | TrainLoss 2.9578 | ValLoss 3.6271 | Val C-index 0.5049
Epoch 008 | TrainLoss 2.9548 | ValLoss 3.6672 | Val C-index 0.5010
Epoch 009 | TrainLoss 2.9262 | ValLoss 3.7203 | Val C-index 0.4951
Epoch 010 | TrainLoss 2.8991 | ValLoss 3.8062 | Val C-index 0.5010
Epoch 011 | TrainLoss 2.9109 | ValLoss 3.8854 | Val C-index 0.5000
Epoch 012 | TrainLoss 2.9016 | ValLoss 4.0835 | Val C-index 0.4961
Epoch 013 | TrainLoss 2.9060 | ValLoss 4.1445 | Val C-index 0.4658
Epoch 014 | TrainLoss 2.8595 | ValLoss 4.1846 | Val C-index 0.5059
Epoch 015 | TrainLoss 2.8606 | ValLoss 4.2467 | Val C-index 0.

['embedding_scaler.joblib']